# Modulo 2 · Lezione 4 — Branch, merge e collaborazione

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Blocco | Cosa imparerai |
|---|--------|----------------|
| 1 | Cos'è un branch | Perché esistono e quando usarli |
| 2 | `git branch` e `git switch` | Creare, elencare e spostarti tra branch |
| 3 | `git merge` | Fondere branch: fast-forward e 3-way |
| 4 | Risoluzione dei conflitti | Cosa fare quando Git non riesce a fondere da solo |
| 5 | Git flow semplificato | La strategia di branching usata dai team |
| 6 | Pull Request | Il processo di revisione del codice |
| 7 | Pratica | Feature branch per il sistema di note |

> 💡 **Idea centrale:** I branch permettono a più linee di lavoro di coesistere senza interferire tra loro. Ogni funzionalità, correzione o esperimento vive nel proprio branch finché non è pronto per essere integrato.

---
## Configurazione dell'ambiente demo

In [ ]:
import os, shutil, subprocess
from pathlib import Path

DEMO = Path.home() / "ramas_demo"

if DEMO.exists():
    shutil.rmtree(DEMO)

DEMO.mkdir()
os.chdir(DEMO)

# Configurare git
!git init -b main
!git config user.name "Alumno Curso"
!git config user.email "alumno@curso.io"

# Creare la base del progetto
Path("app.py").write_text("""\
def saludar(nombre):
    return f"Hola, {nombre}!"

def sumar(a, b):
    return a + b
""", encoding="utf-8")

Path("config.py").write_text("""\
VERSION = "1.0.0"
DEBUG = False
""", encoding="utf-8")

!git add .
!git commit -m "feat: struttura iniziale del progetto"

# Secondo commit
Path("README.md").write_text("# Demo Branch\n", encoding="utf-8")
!git add README.md
!git commit -m "docs: aggiungere README"

print(f"\nRepo demo pronto in: {Path.cwd()}")
!git log --oneline

---
## 1. Cos'è un branch?

Un **branch** è un puntatore mobile a un commit. Quando crei un branch, Git crea solo un nuovo puntatore — non copia i file. Per questo creare branch in Git è praticamente istantaneo.

```
PRIMA (solo main):
A ── B ── C
         ▲
       main (HEAD)

DOPO aver creato il branch 'feature':
A ── B ── C
         ▲ ▲
       main  feature (HEAD)

DOPO un commit su 'feature':
A ── B ── C ── D
         ▲     ▲
       main  feature (HEAD)
```

**`HEAD`** è il puntatore speciale che indica «dove ti trovi adesso». Punta sempre al branch (o commit) attivo.

**A cosa servono i branch?**

- Lavorare su una nuova funzionalità senza toccare il codice che funziona.
- Sistemare un bug urgente mentre sviluppi qualcosa di nuovo.
- Sperimentare senza paura — se non funziona, elimini il branch.
- Collaborare: ogni persona lavora sul proprio branch e poi si fondono.

---
## 2. `git branch` e `git switch` — Gestire i branch

In [ ]:
# Vedere i branch esistenti
print("=== Branch locali ===")
!git branch                  # * segna il branch attuale

print("\n=== Branch con info di tracking ===")
!git branch -vv

print("\n=== Tutti i branch (locali + remoti) ===")
!git branch -a

In [ ]:
# Creare un branch
!git branch feature/login
print("Branch dopo aver creato feature/login:")
!git branch

# Spostarci sul nuovo branch
!git switch feature/login
print("\nOra siamo su:")
!git branch

In [ ]:
# Creare E spostarsi in un solo passaggio (il più comune)
!git switch -c feature/busqueda
print("\nBranch creato e attivato:")
!git branch

# Tornare a main
!git switch main

# Eliminare un branch (quando non ti serve più)
!git branch -d feature/busqueda  # -d: solo se è già stato mergiato
# !git branch -D feature/busqueda # -D: forzare anche se non è mergiato

print("\nBranch finali:")
!git branch

In [ ]:
# Fare commit sul branch feature
!git switch feature/login

Path("login.py").write_text("""\
def login(usuario, contrasena):
    usuarios = {"admin": "secreto"}
    return usuarios.get(usuario) == contrasena
""", encoding="utf-8")

!git add login.py
!git commit -m "feat: aggiungere funzione di login"

Path("login.py").write_text("""\
USUARIOS = {"admin": "secreto", "usuario1": "clave"}

def login(usuario, contrasena):
    return USUARIOS.get(usuario) == contrasena

def logout(usuario):
    print(f"Adiós, {usuario}")
""", encoding="utf-8")

!git add login.py
!git commit -m "feat: aggiungere logout e dizionario utenti"

print("\nCronologia di feature/login:")
!git log --oneline

# Tornare a main: login.py NON esiste qui
!git switch main
print("\nFile in main:")
!ls

---
## 3. `git merge` — Fondere i branch

Quando un branch è pronto, lo fondi con `git merge`. Git ha due strategie:

### Fast-forward merge

Avviene quando `main` non è avanzato da quando è stato creato il branch. Git sposta semplicemente il puntatore in avanti — non serve un commit di merge.

```
PRIMA:                          DOPO (fast-forward):

A ── B ── C ── D                A ── B ── C ── D
         ▲     ▲                             ▲
       main  feature                   main + feature
```

### 3-way merge

Quando entrambi i branch sono avanzati, Git crea un **merge commit** che unisce le due cronologie.

```
PRIMA:                          DOPO (3-way):

A ── B ── C ── E  (main)        A ── B ── C ── E ── M  (M = merge commit)
         └── D    (feature)              └── D ────┘
```

In [ ]:
# Fast-forward merge: main non è avanzato
!git switch main

print("=== PRIMA del merge ===")
!git log --oneline --all --graph

!git merge feature/login

print("\n=== DOPO il merge (fast-forward) ===")
!git log --oneline --all --graph

print("\nOra login.py esiste in main:")
!ls

In [ ]:
# Preparare un 3-way merge: main e feature avanzano in parallelo

# Creare il branch feature/reportes
!git switch -c feature/reportes
Path("reportes.py").write_text("def generar_reporte(): pass\n", encoding="utf-8")
!git add reportes.py
!git commit -m "feat: modulo report (WIP)"

# Nel frattempo, anche main avanza
!git switch main
Path("config.py").write_text("""\
VERSION = "1.1.0"
DEBUG = False
LOG_LEVEL = "INFO"
""", encoding="utf-8")
!git add config.py
!git commit -m "feat: aggiungere LOG_LEVEL in config"

print("Cronologia parallela:")
!git log --oneline --all --graph

In [ ]:
# Merge 3-way: crea un merge commit
!git merge feature/reportes -m "Merge: integrare il modulo report"

print("Cronologia dopo il 3-way merge:")
!git log --oneline --all --graph

---
## 4. Risoluzione dei conflitti

Un **conflitto** si verifica quando due branch modificano la stessa riga dello stesso file. Git non può decidere quale versione conservare — te lo mostra perché sia tu a decidere.

```python
# Così Git segna un conflitto nel file:

<<<<<<< HEAD          # ciò che hai nel tuo branch attuale
VERSION = "1.1.0"
=======               # separatore
VERSION = "2.0.0"    # ciò che viene dal branch che stai mergiando
>>>>>>> feature/v2
```

**Passi per risolvere:**
1. Aprire il file in conflitto.
2. Scegliere cosa conservare (o scrivere qualcosa di diverso).
3. Eliminare i marcatori `<<<<<<<`, `=======`, `>>>>>>>`.
4. `git add <file>`.
5. `git commit`.

In [ ]:
# Provocare un conflitto: entrambi i branch modificano config.py

# Il branch feature/v2 cambia VERSION
!git switch -c feature/v2
Path("config.py").write_text("""\
VERSION = "2.0.0"
DEBUG = False
LOG_LEVEL = "INFO"
MAX_CONEXIONES = 100
""", encoding="utf-8")
!git add config.py
!git commit -m "feat: aggiornare alla versione 2.0.0"

# Anche main cambia config.py sulla stessa riga
!git switch main
Path("config.py").write_text("""\
VERSION = "1.2.0"
DEBUG = False
LOG_LEVEL = "DEBUG"
""", encoding="utf-8")
!git add config.py
!git commit -m "fix: attivare log DEBUG per diagnostica"

print("Cronologia con due branch divergenti:")
!git log --oneline --all --graph

In [ ]:
import subprocess

# Tentare il merge: provocherà un conflitto
r = subprocess.run(["git", "merge", "feature/v2"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)

print("\nFile con marcatori di conflitto:")
print(Path("config.py").read_text())

print("\nStato del repo durante il conflitto:")
!git status

In [ ]:
# Risolvere il conflitto: scegliamo VERSION 2.0.0 e LOG_LEVEL DEBUG
# (combiniamo il meglio di entrambi i branch)
Path("config.py").write_text("""\
VERSION = "2.0.0"
DEBUG = False
LOG_LEVEL = "DEBUG"
MAX_CONEXIONES = 100
""", encoding="utf-8")

print("File risolto:")
print(Path("config.py").read_text())

# Segnare come risolto e completare il merge
!git add config.py
!git commit -m "Merge: integrare v2 con correzione log debug"

print("\nCronologia dopo aver risolto il conflitto:")
!git log --oneline --all --graph

### Se il conflitto è troppo complesso — annullare il merge

```bash
git merge --abort   # torni allo stato precedente al tentativo di merge
```

Usalo quando hai bisogno di tempo per capire le modifiche prima di risolvere.

---
## 5. Git flow semplificato — Strategia dei branch

I team professionali non lavorano direttamente su `main`. Usano un flusso di branch che protegge il codice stabile.

### Il modello più comune (Git flow semplificato)

```
main (produzione — sempre stabile)
 │
 ├── develop (integrazione — codice pronto per essere testato)
 │    ├── feature/login
 │    ├── feature/reportes
 │    └── feature/busqueda
 │
 └── hotfix/bug-critico (correzioni urgenti in produzione)
```

### Flusso di lavoro quotidiano

```bash
# 1. Creare un branch da develop (o da main in progetti piccoli)
git switch develop
git pull                          # assicurarsi di avere l'ultima versione
git switch -c feature/mia-funzionalita

# 2. Lavorare sul branch
# ... modificare i file ...
git add -p                        # aggiungere modifiche selettivamente
git commit -m "feat: descrizione"

# 3. Mantenere il branch aggiornato
git switch develop && git pull
git switch feature/mia-funzionalita
git merge develop                 # oppure git rebase develop

# 4. Quando è pronto: Pull Request (non merge diretto)
git push origin feature/mia-funzionalita
# → aprire la PR su GitHub

# 5. Dopo il merge su GitHub: pulire
git switch develop && git pull
git branch -d feature/mia-funzionalita
```

### Convenzione dei nomi dei branch

| Prefisso | Per | Esempio |
|---------|------|---------|
| `feature/` | Nuova funzionalità | `feature/login-social` |
| `fix/` | Correzione di bug | `fix/errore-404-profilo` |
| `hotfix/` | Bug urgente in produzione | `hotfix/token-scaduto` |
| `docs/` | Solo documentazione | `docs/aggiornare-readme` |
| `refactor/` | Ristrutturare senza cambiare funzionalità | `refactor/auth-module` |

---
## 6. Pull Request — Revisione del codice

Una **Pull Request (PR)** è una richiesta per integrare un branch in un altro. Il suo vero valore non è il merge tecnico (quello Git lo fa da solo) — è la **revisione del codice** che avviene prima del merge.

```
CICLO DI VITA DI UNA PULL REQUEST:

1. Lo sviluppatore fa push del proprio branch su GitHub
   git push origin feature/login

2. Apre la PR su GitHub:
   - Base: develop (o main)
   - Compare: feature/login
   - Titolo e descrizione chiara
   - Assegna i revisori

3. I revisori leggono il codice:
   - Commentano righe specifiche
   - Chiedono modifiche (Request changes)
   - Approvano (Approve)

4. Lo sviluppatore corregge se c'è feedback:
   git commit -m "fix: correggere secondo revisione"
   git push   (la PR si aggiorna automaticamente)

5. Merge della PR (lo fa chi ha i permessi)

6. Eliminare il branch su GitHub (pulsante dopo il merge)
```

### Come scrivere una buona PR

```markdown
## Cosa fa questa PR?
Aggiunge l'autenticazione degli utenti con nome utente e password.

## Perché?
Dobbiamo proteggere le rotte del pannello di amministrazione (issue #42).

## Come l'ho testata?
- [ ] Login corretto restituisce True
- [ ] Password errata restituisce False
- [ ] Utente inesistente restituisce False

## Modifiche
- `login.py`: nuova funzione `login(usuario, contrasena)` e `logout()`
- `config.py`: aggiunto `USUARIOS` (spostare su DB nello sprint 3)
```

---
## 7. Pratica — Feature branch per il sistema di note

Applichiamo il flusso di lavoro professionale al repo `sistema_notas`.

---

### Esercizio 1 — Gestione base dei branch

**Consegna:** Lavora con il repo `sistema_notas`:

1. Crea tre branch: `feature/exportar-csv`, `feature/busqueda-avanzada`, `fix/validacion-fecha`.
2. Mostrali tutti con `git branch`.
3. Scrivi una funzione Python `estado_ramas(ruta)` che usi `subprocess` per mostrare il nome del branch attivo e l'elenco di tutti i branch locali.
4. Elimina il branch `fix/validacion-fecha` (senza fare merge).
5. Mostra la cronologia con `--graph` per vedere la struttura.

In [ ]:
# ✏️ IL TUO CODICE QUI
import os, subprocess
from pathlib import Path

NOTAS = Path.home() / "sistema_notas"
if NOTAS.exists():
    os.chdir(NOTAS)
    print(f"Lavorando in: {Path.cwd()}")
else:
    print("⚠️  Crea prima il repo sistema_notas")


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import os, subprocess
from pathlib import Path

NOTAS = Path.home() / "sistema_notas"
os.chdir(NOTAS)

# 1. Creare i branch
!git branch feature/exportar-csv
!git branch feature/busqueda-avanzada
!git branch fix/validacion-fecha

# 2. Vederli tutti
print("=== Branch ===")
!git branch

# 3. Funzione estado_ramas
def estado_ramas(ruta):
    def git(args):
        r = subprocess.run(["git"] + args, cwd=str(ruta),
                           capture_output=True, text=True)
        return r.stdout.strip()

    activa = git(["branch", "--show-current"])
    todas = git(["branch"]).splitlines()

    print(f"\nBranch attivo: {activa}")
    print(f"Totale branch: {len(todas)}")
    for rama in todas:
        print(f"  {rama.strip()}")

estado_ramas(NOTAS)

# 4. Eliminare branch (senza merge)
!git branch -D fix/validacion-fecha   # -D forza senza merge

# 5. Cronologia con graph
print("\nCronologia:")
!git log --oneline --all --graph
```
</details>

---
### Esercizio 2 — Sviluppare una feature

**Consegna:** Sviluppa la funzionalità di esportare le note in CSV nel proprio branch:

1. Spostati su `feature/exportar-csv`.
2. Crea il file `exportador.py` con una funzione `exportar_csv(notas, ruta_archivo)` che salvi le note in formato CSV con colonne: `id,titulo,contenido,fecha`.
3. Committa con un messaggio appropriato (Conventional Commits).
4. Aggiungi una funzione `importar_csv(ruta_archivo)` nello stesso file.
5. Committa.
6. Fondi `feature/exportar-csv` in `main` con `--no-ff` per forzare un merge commit anche se fosse fast-forward.
7. Controlla la cronologia grafica dopo il merge.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import os
from pathlib import Path

os.chdir(Path.home() / "sistema_notas")

# 1. Spostarsi sul branch
!git switch feature/exportar-csv

# 2. Creare exportador.py
Path("exportador.py").write_text("""\
import csv
from pathlib import Path

def exportar_csv(notas, ruta_archivo):
    ruta = Path(ruta_archivo)
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "titulo", "contenido", "fecha"])
        writer.writeheader()
        for i, nota in enumerate(notas, 1):
            writer.writerow({
                "id": i,
                "titulo": nota.titulo,
                "contenido": nota.contenido,
                "fecha": str(nota.fecha),
            })
    return ruta
""", encoding="utf-8")

# 3. Primo commit
!git add exportador.py
!git commit -m "feat: esportare note in CSV"

# 4. Aggiungere importar_csv
contenido = Path("exportador.py").read_text(encoding="utf-8")
contenido += """
def importar_csv(ruta_archivo):
    from nota import Nota
    notas = []
    with open(ruta_archivo, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for fila in reader:
            n = Nota(fila["titulo"], fila["contenido"])
            notas.append(n)
    return notas
"""
Path("exportador.py").write_text(contenido, encoding="utf-8")

# 5. Secondo commit
!git add exportador.py
!git commit -m "feat: importare note da CSV"

# 6. Merge con --no-ff (crea sempre un merge commit, utile per la cronologia)
!git switch main
!git merge feature/exportar-csv --no-ff -m "Merge: integrare esportazione CSV"

# 7. Cronologia grafica
print("\nCronologia dopo il merge:")
!git log --oneline --all --graph
```

**`--no-ff`** (no fast-forward) crea sempre un merge commit anche quando non sarebbe necessario. Questo è utile per mantenere nella cronologia un punto chiaro in cui è stata integrata ogni feature — facilita l'annullamento di feature intere se necessario.
</details>

---
### Esercizio 3 — Provocare e risolvere un conflitto

**Consegna:** Crea un conflitto controllato e risolvilo:

1. Crea il branch `feature/nueva-config` da `main`.
2. Su quel branch, modifica `gestor.py` aggiungendo la costante `MAX_NOTAS = 100` all'inizio del file.
3. Committa.
4. Torna su `main` e in `gestor.py` aggiungi la costante `MAX_NOTAS = 50` all'inizio (stessa riga, valore diverso).
5. Committa su `main`.
6. Prova a fare il merge di `feature/nueva-config` in `main`.
7. Risolvi il conflitto scegliendo `MAX_NOTAS = 200` (un valore definitivo diverso dagli altri due).
8. Completa il merge.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import os, subprocess
from pathlib import Path

os.chdir(Path.home() / "sistema_notas")

gestor_original = Path("gestor.py").read_text(encoding="utf-8")

# 1-3. Branch feature/nueva-config
!git switch -c feature/nueva-config
Path("gestor.py").write_text("MAX_NOTAS = 100\n" + gestor_original, encoding="utf-8")
!git add gestor.py
!git commit -m "feat: aggiungere limite di note (100)"

# 4-5. Anche main modifica gestor.py
!git switch main
Path("gestor.py").write_text("MAX_NOTAS = 50\n" + gestor_original, encoding="utf-8")
!git add gestor.py
!git commit -m "feat: aggiungere limite di note (50)"

# 6. Tentare il merge — produrrà un conflitto
r = subprocess.run(["git", "merge", "feature/nueva-config"],
                   capture_output=True, text=True)
print(r.stdout + r.stderr)

# Vedere il conflitto
print("Conflitto in gestor.py:")
print(Path("gestor.py").read_text()[:300])

# 7. Risolvere: MAX_NOTAS = 200
contenido_resuelto = Path("gestor.py").read_text(encoding="utf-8")
# Eliminare tutto il blocco di conflitto e mettere il valore deciso
import re
contenido_limpio = re.sub(
    r'<<<<<<< HEAD.*?>>>>>>> feature/nueva-config\n',
    'MAX_NOTAS = 200\n',
    contenido_resuelto,
    flags=re.DOTALL
)
Path("gestor.py").write_text(contenido_limpio, encoding="utf-8")

# 8. Completare il merge
!git add gestor.py
!git commit -m "Merge: risolvere conflitto MAX_NOTAS → 200"

print("\nCronologia finale:")
!git log --oneline --graph
```
</details>

---
### Esercizio 4 — Pubblicare un branch su GitHub

**Consegna:** Esercitati con il flusso completo di una feature branch pubblicata su GitHub:

1. Crea il branch `feature/busqueda-avanzada` in `sistema_notas`.
2. Aggiungi in `gestor.py` un metodo `buscar_avanzada(termino, campo="titulo")` che filtri le note per `titulo` o `contenido` secondo il parametro `campo`.
3. Committa con un messaggio convenzionale.
4. Fai push del branch sul remoto: `git push -u origin feature/busqueda-avanzada`.
5. Verifica su GitHub che il branch appaia e che GitHub ti offra di creare una Pull Request.
6. Scrivi qui (in una cella Markdown) la descrizione della PR che scriveresti.

In [ ]:
# ✏️ IL TUO CODICE QUI


**✏️ Scrivi qui la descrizione della PR:**

```markdown
## Cosa fa questa PR?
...

## Perché?
...

## Come l'ho testata?
- [ ] ...
```

---
### Esercizio 5 (sfida) — Analizzatore di branch

**Consegna:** Scrivi la funzione `analizar_ramas(ruta)` che usi `subprocess` per generare un report completo dello stato dei branch di un repository:

- Branch attivo
- Numero di branch locali e remoti
- Per ogni branch locale: nome, hash dell'ultimo commit, data, messaggio
- Quali branch sono già stati mergiati in `main` (candidati all'eliminazione)
- Quali branch hanno commit che `main` non ha (lavoro in attesa di essere integrato)

Provala con il repo `sistema_notas`.

In [ ]:
# ✏️ IL TUO CODICE QUI
import subprocess
from pathlib import Path


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import subprocess
from pathlib import Path

def analizar_ramas(ruta):
    ruta = Path(ruta).expanduser().resolve()

    def git(args):
        r = subprocess.run(["git"] + args, cwd=str(ruta),
                           capture_output=True, text=True)
        return r.stdout.strip()

    print(f"\n{'═'*60}")
    print(f"  REPORT DEI BRANCH — {ruta.name}")
    print(f"{'═'*60}")

    # Branch attivo
    activa = git(["branch", "--show-current"])
    print(f"\n  Branch attivo: {activa}")

    # Contare i branch
    locales = [l.strip().lstrip("* ") for l in git(["branch"]).splitlines() if l.strip()]
    remotas = [r.strip() for r in git(["branch", "-r"]).splitlines() if r.strip()]
    print(f"  Branch locali:  {len(locales)}")
    print(f"  Branch remoti:  {len(remotas)}")

    # Dettaglio di ogni branch locale
    print(f"\n  {'BRANCH':<30} {'HASH':<8} {'DATA':<12} ULTIMO COMMIT")
    print(f"  {'-'*30} {'-'*8} {'-'*12} {'-'*25}")
    for rama in locales:
        info = git(["log", "-1", f"--format=%h|%ad|%s", "--date=short", rama])
        if info:
            partes = info.split("|", 2)
            hash_, fecha, msg = partes[0], partes[1], partes[2][:25]
        else:
            hash_, fecha, msg = "—", "—", "nessun commit"
        marcador = " ◄" if rama == activa else ""
        print(f"  {rama:<30} {hash_:<8} {fecha:<12} {msg}{marcador}")

    # Branch già mergiati in main
    mergeadas_raw = git(["branch", "--merged", "main"]).splitlines()
    mergeadas = [m.strip().lstrip("* ") for m in mergeadas_raw if m.strip()]
    candidatas = [m for m in mergeadas if m not in ("main", activa)]

    print(f"\n  BRANCH MERGIATI IN main (candidati all'eliminazione):")
    if candidatas:
        for c in candidatas:
            print(f"    git branch -d {c}")
    else:
        print("    Nessuno")

    # Branch con lavoro in attesa di essere integrato
    print(f"\n  BRANCH CON COMMIT IN ATTESA DI INTEGRAZIONE IN main:")
    pendientes = False
    for rama in locales:
        if rama == "main":
            continue
        adelantados = git(["rev-list", "main.."+rama, "--count"])
        if adelantados and adelantados != "0":
            print(f"    {rama}: {adelantados} commit avanti rispetto a main")
            pendientes = True
    if not pendientes:
        print("    Tutti i branch sono integrati in main")

    print(f"\n{'═'*60}")


analizar_ramas("~/sistema_notas")
```
</details>

---
## Riepilogo della lezione

```bash
# ─── Branch ────────────────────────────────────────────────
git branch                        # elencare i branch
git branch nome                   # creare un branch
git switch nome                   # passare a un branch
git switch -c nome                # creare + passare
git branch -d nome                # eliminare (se è mergiato)
git branch -D nome                # eliminare (forzare)
git branch -vv                    # vedere con info di tracking
git branch --merged main          # branch già integrati

# ─── Merge ─────────────────────────────────────────────────
git merge feature/login           # merge fast-forward (se possibile)
git merge feature/login --no-ff   # crea sempre un merge commit
git merge --abort                 # annullare merge con conflitti

# ─── Conflitti ─────────────────────────────────────────────
# 1. git status              → vedere quali file hanno conflitti
# 2. modificare il file      → rimuovere <<<, ===, >>> e scegliere
# 3. git add <file>          → segnare come risolto
# 4. git commit              → completare il merge

# ─── Branch remoti ─────────────────────────────────────────
git push -u origin feature/login  # pubblicare il branch su GitHub
git branch -r                     # vedere i branch remoti
git fetch --prune                 # sincronizzare i branch remoti eliminati

# ─── Ispezione ─────────────────────────────────────────────
git log --oneline --all --graph   # cronologia di tutti i branch
git rev-list main..feature --count # quanti commit avanti
```

## Cosa vediamo nella prossima lezione?

La **Lezione 5** è il progetto integratore: versioniamo il sistema di biblioteca della Lezione 12 del Modulo 1, applicando tutto quello che abbiamo imparato — branch, commit convenzionali, cronologia pulita, tag di versione e pubblicazione su GitHub.